# Bearing fault diagnosis with STFT, DWT and a 1‑D CNN

This notebook adapts the instructor's two-block Conv1D example to classify healthy, inner-race, ball and outer-race vibration recordings. Each model input is a time sequence of numeric STFT and DWT features. The original plots are audited in [the project notes](../docs/stft_dwt_audit.md).

Install the project's dependencies from the repository root with `python -m pip install -r requirements.txt`. Use the same Python environment for the notebook kernel. The cloned repository includes the input recordings.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "Datasets" / "RAW").is_dir():
    raise RuntimeError("Open Jupyter from the repository root or notebooks folder.")
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

from src.data_pipeline import CLASSES
from src.stft_dwt import DWT_BANDS, FRAME_HOP, FRAME_SAMPLES, FREQUENCY_BINS, SAMPLE_RATE
from src.train_cnn import RAW_DIR, MODEL_PATH, prepare_uploaded_cohort, load_partitions, fit_model, evaluate_test

## 1. Prepare the recordings and inspect the features

The locked cohort contains 16 drive-end recordings. Training uses 0 and 1 HP, validation uses 2 HP, and the final test uses 3 HP. All windows from a recording stay in its assigned split. The prepared windows are centered, retain their amplitude, and share a target rate of 12 kHz.

In [ ]:
with TemporaryDirectory() as work:
    summary = prepare_uploaded_cohort(RAW_DIR, work)
    partitions = load_partitions(work)
    with np.load(Path(work) / "validation.npz", allow_pickle=False) as data:
        examples = [np.flatnonzero(data["labels"] == label)[0] for label in CLASSES.values()]
        example_waves = data["feature_windows"][examples]
        example_ids = data["record_ids"][examples]

for split, (features, labels, groups) in partitions.items():
    print(split, features.shape, np.unique(groups).size, "recordings")

Each 2,048-sample window produces 15 time frames. The STFT uses 256-sample Hann frames with a 128-sample hop. On the same sample intervals a four-level db4 DWT produces five subband features (mean squared coefficients). The CNN sees 134 channels per time frame, consisting of 129 STFT bins and five DWT bands. The plots below show the exact pre-normalization model inputs from one validation window per class. DWT band names indicate approximate subbands because the wavelet filters have transition regions.

In [ ]:
feature_examples = partitions["validation"][0][examples]
stft_examples = feature_examples[:, :, :FREQUENCY_BINS]
dwt_examples = feature_examples[:, :, FREQUENCY_BINS:]
samples_ms = np.arange(example_waves.shape[1]) * 1000 / SAMPLE_RATE
frame_ms = (FRAME_SAMPLES / 2 + FRAME_HOP * np.arange(feature_examples.shape[1])) * 1000 / SAMPLE_RATE
frequency_khz = np.fft.rfftfreq(FRAME_SAMPLES, d=1 / SAMPLE_RATE) / 1000

fig, axes = plt.subplots(len(CLASSES), 3, figsize=(13, 10), layout="constrained")
for i, (label, record) in enumerate(zip(CLASSES, example_ids)):
    wave_ax, stft_ax, dwt_ax = axes[i]
    wave_ax.plot(samples_ms, example_waves[i], lw=0.7)
    wave_ax.set(title=f"{label.replace('_', ' ').title()} · {record}", ylabel="Signal amplitude")
    stft_plot = stft_ax.pcolormesh(
        frame_ms, frequency_khz, stft_examples[i].T, shading="auto",
        vmin=stft_examples.min(), vmax=stft_examples.max(),
    )
    stft_ax.set(ylabel="Frequency (kHz)", ylim=(0, SAMPLE_RATE / 2000))
    dwt_plot = dwt_ax.pcolormesh(
        frame_ms, np.arange(len(DWT_BANDS)), dwt_examples[i].T, shading="auto",
        vmin=dwt_examples.min(), vmax=dwt_examples.max(),
    )
    dwt_ax.set_yticks(np.arange(len(DWT_BANDS)), DWT_BANDS)
    dwt_ax.set_ylabel("Wavelet band")
axes[0, 1].set_title("STFT log power")
axes[0, 2].set_title("DWT mean squared coefficients")
for ax in axes[-1]:
    ax.set_xlabel("Time (ms)")
fig.colorbar(stft_plot, ax=axes[:, 1], label="dB")
fig.colorbar(dwt_plot, ax=axes[:, 2], label="dB")
plt.show()

## 2. Train on the training recordings

The model keeps the instructor's two Conv1D and MaxPooling1D blocks, dense layer, dropout and softmax. Its normalization uses only the training features. Validation loss selects the weights. Integer labels use sparse categorical cross-entropy.

In [ ]:
model, history = fit_model(partitions)
model.summary()

## 3. Evaluate the held-out recordings

The window-level metrics describe overlapping windows. The recording-level metrics average window probabilities per original recording and give each recording one vote. There are only four test recordings (one per class) at 3 HP. This is a pilot on the 0.007-inch CWRU cohort.

In [ ]:
metrics = evaluate_test(model, partitions["test"])
for unit in ("windows", "recordings"):
    result = metrics[unit]
    print(f"{unit}: n={result['count']}, accuracy={result['accuracy']:.3f}, macro-F1={result['macro_f1']:.3f}")
for prediction in metrics["record_predictions"]:
    print(prediction)

ConfusionMatrixDisplay(
    confusion_matrix=np.asarray(metrics["recordings"]["confusion_matrix"]),
    display_labels=metrics["class_order"],
).plot(xticks_rotation=30, cmap="Blues")
plt.show()

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(MODEL_PATH)
print(f"Saved {MODEL_PATH}")